In [6]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parents[1]
DATASET_DIR = PROJECT_ROOT / "data" / "raw" / "COde-Dataset"
CSV_PATH = DATASET_DIR / "complete_dataset.csv"

print("Working directory :", Path.cwd())
print("Project root      :", PROJECT_ROOT)
print("Dataset directory :", DATASET_DIR)
print("CSV path          :", CSV_PATH)
print("CSV exists        :", CSV_PATH.exists())

assert CSV_PATH.exists(), f"Dataset not found: {CSV_PATH}"

Working directory : /home/ubuntu/Projects/thesis-code/notebooks/Thesis Note
Project root      : /home/ubuntu/Projects/thesis-code
Dataset directory : /home/ubuntu/Projects/thesis-code/data/raw/COde-Dataset
CSV path          : /home/ubuntu/Projects/thesis-code/data/raw/COde-Dataset/complete_dataset.csv
CSV exists        : True


In [7]:
import pandas as pd
import re

df = pd.read_csv(
    CSV_PATH,
    usecols=["patient_id", "checkup_id", "anomalies_en"],
)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

s = df["anomalies_en"].fillna("").astype(str).str.strip()

patterns = {
    "Caries": r"\bcaries\b",
    "Gingivitis": r"\bgingivitis\b",
    "Malocclusion": r"\bmalocclusion\b",
    "Pulpitis": r"\bpulpitis\b",
    "Tooth Loss": r"\btooth loss\b",
    "Tooth Structure Loss": r"\btooth structure loss\b",
}

print("=" * 80)
print("COde LABEL RECONSTRUCTION AUDIT")
print("=" * 80)

print(f"Total visits: {len(df):,}")
print(f"Non-empty anomalies_en: {(s != '').sum():,}")
print(f"Empty anomalies_en: {(s == '').sum():,}")

print("\n" + "=" * 80)
print("DIRECT SIX-CLASS MATCHING")
print("=" * 80)

matched = pd.DataFrame(index=df.index)

for label, pattern in patterns.items():
    matched[label] = s.str.contains(
        pattern,
        case=False,
        regex=True,
    )

    print(
        f"{label:<25}"
        f"{matched[label].sum():>6,} "
        f"({matched[label].mean() * 100:6.2f}%)"
    )

matched_count = matched.sum(axis=1)

print("\n" + "=" * 80)
print("NUMBER OF MATCHED CLASSES PER VISIT")
print("=" * 80)

print(
    matched_count
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n" + "=" * 80)
print("VISITS COVERED BY AT LEAST ONE OF SIX CLASSES")
print("=" * 80)

covered = matched_count > 0

print(f"Covered: {(covered).sum():,}")
print(f"Not covered: {(~covered).sum():,}")
print(f"Coverage: {covered.mean() * 100:.2f}%")

print("\n" + "=" * 80)
print("SAMPLES WITH NON-EMPTY LABEL BUT NO SIX-CLASS MATCH")
print("=" * 80)

unmatched = df.loc[
    (s != "") & (~covered),
    ["checkup_id", "patient_id", "anomalies_en"]
]

print(f"Count: {len(unmatched):,}")

print("\nTop 50 unmatched values:")

print(
    unmatched["anomalies_en"]
    .value_counts()
    .head(50)
    .to_string()
)

print("\n" + "=" * 80)
print("SAMPLES WITH MULTIPLE SIX-CLASS LABELS")
print("=" * 80)

multi = df.loc[
    matched_count > 1,
    ["checkup_id", "patient_id", "anomalies_en"]
].copy()

print(f"Count: {len(multi):,}")

print("\nTop 30 combinations:")

print(
    multi["anomalies_en"]
    .value_counts()
    .head(30)
    .to_string()
)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

Rows: 8775
Columns: ['checkup_id', 'patient_id', 'anomalies_en']
COde LABEL RECONSTRUCTION AUDIT
Total visits: 8,775
Non-empty anomalies_en: 7,654
Empty anomalies_en: 1,121

DIRECT SIX-CLASS MATCHING
Caries                      955 ( 10.88%)
Gingivitis                1,987 ( 22.64%)
Malocclusion              2,636 ( 30.04%)
Pulpitis                    443 (  5.05%)
Tooth Loss                  309 (  3.52%)
Tooth Structure Loss      1,042 ( 11.87%)

NUMBER OF MATCHED CLASSES PER VISIT
0    2090
1    6058
2     576
3      42
4       9

VISITS COVERED BY AT LEAST ONE OF SIX CLASSES
Covered: 6,685
Not covered: 2,090
Coverage: 76.18%

SAMPLES WITH NON-EMPTY LABEL BUT NO SIX-CLASS MATCH
Count: 969

Top 50 unmatched values:
anomalies_en
Dental Crowding                                                                                               111
Periapical Periodontitis                                                                                       82
Convex Profile,Dental Crowding  

In [8]:
from pathlib import Path
import pandas as pd
from collections import Counter, defaultdict

# ============================================================
# PATH SETUP
# ============================================================

PROJECT_ROOT = Path.cwd().resolve().parents[1]
PATH = PROJECT_ROOT / "data/raw/COde-Dataset/complete_dataset.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset:", PATH)
print("Dataset exists:", PATH.exists())

if not PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {PATH}")


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(
    PATH,
    usecols=["patient_id", "checkup_id", "anomalies_en"]
)

s = df["anomalies_en"].fillna("").astype(str).str.strip()


# ============================================================
# ATOMIC LABEL INVENTORY
# ============================================================

label_visits = Counter()
label_patients = defaultdict(set)

for idx, value in s.items():

    if not value:
        continue

    labels = [
        x.strip()
        for x in value.split(",")
        if x.strip()
    ]

    # One label counted once per visit
    for label in set(labels):
        label_visits[label] += 1
        label_patients[label].add(df.loc[idx, "patient_id"])


# ============================================================
# BASIC SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("ATOMIC LABEL INVENTORY")
print("=" * 100)

print(f"Total visits: {len(df):,}")
print(f"Visits with anomalies_en: {(s != '').sum():,}")
print(f"Visits without anomalies_en: {(s == '').sum():,}")
print(f"Unique atomic labels: {len(label_visits):,}")


# ============================================================
# ALL LABELS
# ============================================================

print("\n" + "=" * 100)
print("ALL ATOMIC LABELS")
print("=" * 100)

print(
    f"{'Rank':<6}"
    f"{'Label':<65}"
    f"{'Visits':>10}"
    f"{'Patients':>10}"
)

print("-" * 100)

for rank, (label, count) in enumerate(
    label_visits.most_common(),
    start=1
):
    patients = len(label_patients[label])

    print(
        f"{rank:<6}"
        f"{label[:63]:<65}"
        f"{count:>10,}"
        f"{patients:>10,}"
    )


# ============================================================
# THRESHOLD SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("THRESHOLD SUMMARY")
print("=" * 100)

thresholds = [
    10, 20, 30, 50, 75,
    100, 150, 200, 300,
    500, 750, 1000
]

print(
    f"{'Threshold':<15}"
    f"{'Labels':>10}"
    f"{'Occurrences':>18}"
)

print("-" * 50)

for t in thresholds:

    labels = [
        label
        for label, count in label_visits.items()
        if count >= t
    ]

    occurrences = sum(
        label_visits[label]
        for label in labels
    )

    print(
        f">= {t:<12}"
        f"{len(labels):>10,}"
        f"{occurrences:>18,}"
    )


# ============================================================
# HIGH-FREQUENCY LABELS
# ============================================================

print("\n" + "=" * 100)
print("HIGH-FREQUENCY LABELS (>= 50 VISITS)")
print("=" * 100)

high_freq = [
    (label, count, len(label_patients[label]))
    for label, count in label_visits.most_common()
    if count >= 50
]

print(
    f"{'Rank':<6}"
    f"{'Label':<65}"
    f"{'Visits':>10}"
    f"{'Patients':>10}"
)

print("-" * 100)

for rank, (label, visits, patients) in enumerate(
    high_freq,
    start=1
):
    print(
        f"{rank:<6}"
        f"{label[:63]:<65}"
        f"{visits:>10,}"
        f"{patients:>10,}"
    )


# ============================================================
# TOP 30 — CLEAN VIEW
# ============================================================

print("\n" + "=" * 100)
print("TOP 30 LABELS — CLEAN VIEW")
print("=" * 100)

for rank, (label, count) in enumerate(
    label_visits.most_common(30),
    start=1
):
    print(
        f"{rank:>2}. "
        f"{label} "
        f"({count:,} visits, "
        f"{len(label_patients[label]):,} patients)"
    )


print("\n" + "=" * 100)
print("DONE")
print("=" * 100)

Project root: /home/ubuntu/Projects/thesis-code
Dataset: /home/ubuntu/Projects/thesis-code/data/raw/COde-Dataset/complete_dataset.csv
Dataset exists: True

ATOMIC LABEL INVENTORY
Total visits: 8,775
Visits with anomalies_en: 7,654
Visits without anomalies_en: 1,121
Unique atomic labels: 117

ALL ATOMIC LABELS
Rank  Label                                                                Visits  Patients
----------------------------------------------------------------------------------------------------
1     Gingivitis                                                            1,987     1,691
2     Class II Malocclusion                                                 1,825     1,498
3     Dental Crowding                                                       1,148     1,066
4     Tooth Structure Loss                                                  1,042       831
5     Dental Caries                                                           955       810
6     Convex Profile                

In [9]:
import json
from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path.cwd().resolve().parents[1]
DATASET_DIR = PROJECT_ROOT / "data/raw/COde-Dataset"

TRAIN_CLS = DATASET_DIR / "train-cls.json"
TEST_CLS = DATASET_DIR / "test_cls.json"

print("TRAIN:", TRAIN_CLS.exists(), TRAIN_CLS)
print("TEST :", TEST_CLS.exists(), TEST_CLS)


def extract_cls_labels(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    labels = []

    for sample in data:
        messages = sample.get("messages", [])

        for message in messages:
            if message.get("role") == "assistant":
                label = message.get("content", "").strip()
                if label:
                    labels.append(label)
                break

    return data, labels


train_cls, train_labels = extract_cls_labels(TRAIN_CLS)
test_cls, test_labels = extract_cls_labels(TEST_CLS)


print("\n" + "=" * 80)
print("COde ORIGINAL CLASSIFICATION BENCHMARK")
print("=" * 80)

print(f"Train samples: {len(train_cls):,}")
print(f"Test samples : {len(test_cls):,}")

print("\nTrain labels:")
for label, count in Counter(train_labels).most_common():
    print(f"  {label:<25} {count:>6,}")

print("\nTest labels:")
for label, count in Counter(test_labels).most_common():
    print(f"  {label:<25} {count:>6,}")

print("\nUnique train labels:", len(set(train_labels)))
print("Unique test labels :", len(set(test_labels)))

print("\nTrain ∩ Test labels:")
print(sorted(set(train_labels) & set(test_labels)))

TRAIN: True /home/ubuntu/Projects/thesis-code/data/raw/COde-Dataset/train-cls.json
TEST : True /home/ubuntu/Projects/thesis-code/data/raw/COde-Dataset/test_cls.json

COde ORIGINAL CLASSIFICATION BENCHMARK
Train samples: 14,696
Test samples : 1,793

Train labels:
  牙龈炎                        1,358
  Gingivitis                 1,347
  Class II Malocclusion      1,114
  Ⅱ类错颌                       1,104
  牙体缺损                         692
  Tooth Structure Loss         654
  Class I Malocclusion         254
  Ⅰ类错颌                         235
  龋齿                           181
  Dental Caries                165
  Pulpitis                     151
  牙髓炎                          149
  Dental Crowding              121
  Tooth Loss                    99
  Ⅲ类错颌                          97
  Class III Malocclusion        90
  牙齿缺失                          88
  Periapical Periodontitis      87
  根尖周炎                          85
  Periodontitis,Tooth Mobility,Gingivitis     69
  牙周炎,牙龈炎              

In [11]:
# ============================================================
# ATOMIC LABEL DISTRIBUTION ACROSS PATIENT-LEVEL SPLIT
# ============================================================

from pathlib import Path
from collections import defaultdict
import pandas as pd


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().resolve().parents[1]

DATASET_PATH = (
    PROJECT_ROOT
    / "data/raw/COde-Dataset/complete_dataset.csv"
)

PATIENT_SPLIT_PATH = (
    PROJECT_ROOT
    / "results/patient_level_split/patient_split.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results/label_reconstruction"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Dataset       :", DATASET_PATH)
print("Patient split :", PATIENT_SPLIT_PATH)

assert DATASET_PATH.exists(), (
    f"Dataset not found: {DATASET_PATH}"
)

assert PATIENT_SPLIT_PATH.exists(), (
    f"Patient split not found: {PATIENT_SPLIT_PATH}"
)


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(
    DATASET_PATH,
    usecols=[
        "patient_id",
        "checkup_id",
        "anomalies_en"
    ]
)

patient_split = pd.read_csv(
    PATIENT_SPLIT_PATH,
    usecols=[
        "patient_id",
        "split"
    ]
)


print(f"\nDataset visits : {len(df):,}")
print(
    f"Dataset patients: "
    f"{df['patient_id'].nunique():,}"
)

print(
    f"Split patients  : "
    f"{patient_split['patient_id'].nunique():,}"
)


# ============================================================
# ATTACH PATIENT-LEVEL SPLIT
# ============================================================

df = df.merge(
    patient_split,
    on="patient_id",
    how="left",
    validate="many_to_one"
)


# ============================================================
# VALIDATE SPLIT ASSIGNMENT
# ============================================================

missing_split = df["split"].isna().sum()

print(
    f"\nVisits without split assignment: "
    f"{missing_split:,}"
)

assert missing_split == 0, (
    "Some visits do not have a patient-level split assignment."
)


print("\nSplit visit counts:")
print(
    df["split"]
    .value_counts()
    .sort_index()
)


# ============================================================
# RECONSTRUCT ATOMIC LABELS
# ============================================================

label_visits = defaultdict(set)
label_patients = defaultdict(set)

for _, row in df.iterrows():

    value = row["anomalies_en"]

    if pd.isna(value):
        continue

    value = str(value).strip()

    if not value:
        continue

    labels = {
        x.strip()
        for x in value.split(",")
        if x.strip()
    }

    for label in labels:

        # Unique visits carrying this label
        label_visits[label].add(
            row["checkup_id"]
        )

        # Unique patients carrying this label
        label_patients[label].add(
            row["patient_id"]
        )


# ============================================================
# BUILD LABEL × SPLIT TABLE
# ============================================================

records = []

for label in sorted(label_visits):

    label_visit_ids = label_visits[label]

    label_rows = df[
        df["checkup_id"].isin(label_visit_ids)
    ]

    train_count = (
        label_rows["split"] == "train"
    ).sum()

    validation_count = (
        label_rows["split"] == "validation"
    ).sum()

    test_count = (
        label_rows["split"] == "test"
    ).sum()

    total_count = (
        train_count
        + validation_count
        + test_count
    )

    records.append(
        {
            "label": label,
            "train_visits": int(train_count),
            "validation_visits": int(validation_count),
            "test_visits": int(test_count),
            "total_visits": int(total_count),
            "patients": len(label_patients[label])
        }
    )


label_split_table = (
    pd.DataFrame(records)
    .sort_values(
        "total_visits",
        ascending=False
    )
    .reset_index(drop=True)
)

label_split_table.insert(
    0,
    "rank",
    range(1, len(label_split_table) + 1)
)


# ============================================================
# ADD SPLIT PERCENTAGES
# ============================================================

label_split_table["train_pct"] = (
    label_split_table["train_visits"]
    / label_split_table["total_visits"]
    * 100
)

label_split_table["validation_pct"] = (
    label_split_table["validation_visits"]
    / label_split_table["total_visits"]
    * 100
)

label_split_table["test_pct"] = (
    label_split_table["test_visits"]
    / label_split_table["total_visits"]
    * 100
)


# ============================================================
# DISPLAY
# ============================================================

print("\n" + "=" * 120)
print("ATOMIC LABEL DISTRIBUTION ACROSS PATIENT-LEVEL SPLIT")
print("=" * 120)

display(
    label_split_table.style
    .format(
        {
            "train_pct": "{:.1f}%",
            "validation_pct": "{:.1f}%",
            "test_pct": "{:.1f}%"
        }
    )
)


# ============================================================
# SAVE FULL TABLE
# ============================================================

OUTPUT_PATH = (
    OUTPUT_DIR
    / "atomic_label_patient_level_split.csv"
)

label_split_table.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    f"\nSaved full table to:\n{OUTPUT_PATH}"
)


# ============================================================
# QUICK SUMMARY
# ============================================================

print("\n" + "=" * 120)
print("LABEL DISTRIBUTION SUMMARY")
print("=" * 120)

print(
    f"Total atomic labels : "
    f"{len(label_split_table):,}"
)

print(
    f"Labels with >= 10 train visits : "
    f"{(label_split_table['train_visits'] >= 10).sum():,}"
)

print(
    f"Labels with >= 20 train visits : "
    f"{(label_split_table['train_visits'] >= 20).sum():,}"
)

print(
    f"Labels with >= 30 train visits : "
    f"{(label_split_table['train_visits'] >= 30).sum():,}"
)

print(
    f"Labels with >= 50 train visits : "
    f"{(label_split_table['train_visits'] >= 50).sum():,}"
)

print(
    f"Labels with >= 100 train visits : "
    f"{(label_split_table['train_visits'] >= 100).sum():,}"
)

print("\nDONE")

Dataset       : /home/ubuntu/Projects/thesis-code/data/raw/COde-Dataset/complete_dataset.csv
Patient split : /home/ubuntu/Projects/thesis-code/results/patient_level_split/patient_split.csv

Dataset visits : 8,775
Dataset patients: 4,800
Split patients  : 4,800

Visits without split assignment: 0

Split visit counts:
split
test          1316
train         6129
validation    1330
Name: count, dtype: int64

ATOMIC LABEL DISTRIBUTION ACROSS PATIENT-LEVEL SPLIT


,rank,label,train_visits,validation_visits,test_visits,total_visits,patients,train_pct,validation_pct,test_pct
0,1,Gingivitis,1357,328,302,1987,1691,68.3%,16.5%,15.2%
1,2,Class II Malocclusion,1299,264,262,1825,1498,71.2%,14.5%,14.4%
2,3,Dental Crowding,808,171,169,1148,1066,70.4%,14.9%,14.7%
3,4,Tooth Structure Loss,742,157,143,1042,831,71.2%,15.1%,13.7%
4,5,Dental Caries,660,139,156,955,810,69.1%,14.6%,16.3%
5,6,Convex Profile,525,108,113,746,739,70.4%,14.5%,15.1%
6,7,Mandibular Skeletal Asymmetry,373,74,77,524,524,71.2%,14.1%,14.7%
7,8,Periodontitis,347,74,59,480,404,72.3%,15.4%,12.3%
8,9,Class III Malocclusion,327,59,76,462,389,70.8%,12.8%,16.5%
9,10,Pulpitis,323,45,75,443,364,72.9%,10.2%,16.9%



Saved full table to:
/home/ubuntu/Projects/thesis-code/results/label_reconstruction/atomic_label_patient_level_split.csv

LABEL DISTRIBUTION SUMMARY
Total atomic labels : 117
Labels with >= 10 train visits : 46
Labels with >= 20 train visits : 34
Labels with >= 30 train visits : 31
Labels with >= 50 train visits : 24
Labels with >= 100 train visits : 18

DONE


# 04 — Label Reconstruction Analysis

## Objective

The objective of this analysis was to investigate the diagnostic labels available in the COde dataset and determine whether a reliable classification label space could be reconstructed from the `anomalies_en` field.

The original dataset contains a large number of heterogeneous anomaly labels. Therefore, before constructing the classification benchmark, the label distribution, frequency, patient coverage, and compatibility with the original COde classification benchmark were examined.

---

## 1. Dataset-Level Label Audit

The analysis was performed on:

```text
data/raw/COde-Dataset/complete_dataset.csv
```

The dataset contains:

* **8,775 visits**
* **4,800 unique patients**
* **7,654 visits with non-empty `anomalies_en`**
* **1,121 visits without `anomalies_en`**

Direct matching against the six broad diagnostic categories used in the initial reconstruction audit showed that these six categories do not fully represent the label space of the dataset.

The atomic-label analysis was therefore performed by parsing the individual comma-separated labels contained in `anomalies_en`.

---

## 2. Atomic Label Inventory

Each non-empty `anomalies_en` value was split into individual labels.

A label was counted **once per visit**, even if the same label appeared more than once within that visit.

For each atomic label, two statistics were calculated:

1. Number of visits containing the label
2. Number of unique patients containing the label

The analysis identified:

> **117 unique atomic labels**

The most frequent labels were:

| Rank | Label                         | Visits | Patients |
| ---: | ----------------------------- | -----: | -------: |
|    1 | Gingivitis                    |  1,987 |    1,691 |
|    2 | Class II Malocclusion         |  1,825 |    1,498 |
|    3 | Dental Crowding               |  1,148 |    1,066 |
|    4 | Tooth Structure Loss          |  1,042 |      831 |
|    5 | Dental Caries                 |    955 |      810 |
|    6 | Convex Profile                |    746 |      739 |
|    7 | Mandibular Skeletal Asymmetry |    524 |      524 |
|    8 | Periodontitis                 |    480 |      404 |
|    9 | Class III Malocclusion        |    462 |      389 |
|   10 | Pulpitis                      |    443 |      364 |
|   11 | Deep Overbite                 |    421 |      412 |
|   12 | Class I Malocclusion          |    359 |      332 |
|   13 | Tooth Loss                    |    309 |      268 |

The frequency decreases substantially after these labels, with many remaining labels having relatively few observations.

---

## 3. Selection of Classification Label Space

Based on the observed distribution, the first **13 highest-frequency labels** were selected as the candidate classification label space.

The selected labels are:

1. Gingivitis
2. Class II Malocclusion
3. Dental Crowding
4. Tooth Structure Loss
5. Dental Caries
6. Convex Profile
7. Mandibular Skeletal Asymmetry
8. Periodontitis
9. Class III Malocclusion
10. Pulpitis
11. Deep Overbite
12. Class I Malocclusion
13. Tooth Loss

The cutoff was selected because labels below this point become progressively less frequent, making them less suitable for a stable multi-class benchmark under the current dataset size.

This selection is treated as a **working label space** for the subsequent reconstruction and benchmarking pipeline. It does not imply that the remaining labels are clinically unimportant.

---

## 4. Original COde Classification Benchmark

The original COde classification files were also inspected:

```text
data/raw/COde-Dataset/train-cls.json
data/raw/COde-Dataset/test-cls.json
```

The original benchmark contains:

* **14,696 training samples**
* **1,793 test samples**

The original benchmark uses a mixture of English and Chinese label names. The train/test label intersection was also examined.

This analysis showed that the original classification benchmark represents only a subset of the diagnostic information available in `anomalies_en`.

Therefore, the original benchmark split was **not adopted as the thesis train/validation/test split**.

Instead, the thesis uses the independently generated **patient-level split** described below.

---

## 5. Patient-Level Split

A separate patient-level split was previously generated from:

```text
data/raw/COde-Dataset/complete_dataset.csv
```

using:

* Split unit: `patient_id`
* Random seed: `42`
* Train: **70%**
* Validation: **15%**
* Test: **15%**

The split was stratified at the patient level using:

* diagnosis availability
* radiograph availability

The resulting patient counts are:

| Split      |  Patients |
| ---------- | --------: |
| Train      |     3,360 |
| Validation |       720 |
| Test       |       720 |
| **Total**  | **4,800** |

The resulting visit counts are:

| Split      |    Visits |
| ---------- | --------: |
| Train      |     6,129 |
| Validation |     1,330 |
| Test       |     1,316 |
| **Total**  | **8,775** |

The patient-level split is stored in:

```text
results/patient_level_split/patient_split.csv
```

and the corresponding visit-level mapping in:

```text
results/patient_level_split/visit_split.csv
```

---

## 6. Label Distribution Across the Patient-Level Split

The selected labels were mapped onto the independently generated patient-level split.

For the 13 selected labels, the resulting visit distribution was examined across train, validation, and test sets.

The largest classes show relatively close agreement with the intended 70/15/15 split.

For example:

| Label                         | Train | Validation | Test | Total |
| ----------------------------- | ----: | ---------: | ---: | ----: |
| Gingivitis                    | 1,357 |        328 |  302 | 1,987 |
| Class II Malocclusion         | 1,299 |        264 |  262 | 1,825 |
| Dental Crowding               |   808 |        171 |  169 | 1,148 |
| Tooth Structure Loss          |   742 |        157 |  143 | 1,042 |
| Dental Caries                 |   660 |        139 |  156 |   955 |
| Convex Profile                |   525 |        108 |  113 |   746 |
| Mandibular Skeletal Asymmetry |   373 |         74 |   77 |   524 |
| Periodontitis                 |   347 |         74 |   59 |   480 |
| Class III Malocclusion        |   327 |         59 |   76 |   462 |
| Pulpitis                      |   323 |         45 |   75 |   443 |
| Deep Overbite                 |   284 |         69 |   68 |   421 |
| Class I Malocclusion          |   251 |         56 |   52 |   359 |
| Tooth Loss                    |   210 |         42 |   57 |   309 |

The deviations from exactly 70/15/15 are expected because the split was performed at the **patient level**, rather than independently assigning individual visits.

This is intentional: preserving patient-level independence is more important than obtaining exact per-label visit ratios.

---

## 7. Important Dataset Property: Multi-Label Nature

The `anomalies_en` field is inherently multi-label.

A single visit may contain multiple diagnostic labels. Therefore, the reconstructed label representation must preserve the original multi-label information rather than arbitrarily assigning one diagnosis to each visit.

Consequently, the reconstruction pipeline should distinguish between:

* the complete reconstructed label set of a visit
* membership in the selected 13-label classification space
* samples suitable for a specific classification benchmark

This distinction will be maintained in the subsequent dataset-integration stage.

---

## 8. Current Decision

At this stage, the following decisions have been made:

* Use `complete_dataset.csv` as the authoritative dataset source.
* Do not reuse the original COde train/test split for the thesis benchmark.
* Use the independently generated patient-level split.
* Reconstruct labels from `anomalies_en`.
* Preserve the multi-label nature of the source annotations.
* Use the top 13 most frequent labels as the current candidate classification label space.
* Keep the remaining atomic labels documented rather than deleting them from the underlying dataset.

---

## 9. Next Step

The next milestone is to integrate the reconstructed labels with the patient-level split and the original dataset metadata.

The target pipeline is:

```text
complete_dataset.csv
        │
        ├── patient_id
        ├── checkup_id
        ├── clinical text
        ├── photographs
        ├── radiographs
        └── anomalies_en
                │
                ▼
        Label Reconstruction
                │
                ▼
        Reconstructed Labels
                │
                + patient_split.csv
                │
                ▼
        Integrated Labeled Dataset
                │
                ├── train
                ├── validation
                └── test
```

The next implementation stage will produce a reproducible label-reconstruction artifact and connect it to the existing patient-level split.

**No model training or image loading is performed at this stage.**


In [14]:
# ============================================================
# FINAL LABELED PATIENT-LEVEL DATASET — NOTEBOOK OUTPUT
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve().parents[1]

LABELED_DATASET_PATH = (
    PROJECT_ROOT
    / "results"
    / "labeled_patient_level_dataset"
    / "labeled_dataset.csv"
)

print("Labeled dataset :", LABELED_DATASET_PATH)
print("Exists          :", LABELED_DATASET_PATH.exists())

assert LABELED_DATASET_PATH.exists(), (
    f"Labeled dataset not found: {LABELED_DATASET_PATH}"
)

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

labeled_df = pd.read_csv(LABELED_DATASET_PATH)

# ------------------------------------------------------------
# BASIC DATASET SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("FINAL LABELED PATIENT-LEVEL DATASET")
print("=" * 100)

print(f"Rows    : {len(labeled_df):,}")
print(f"Columns : {len(labeled_df.columns):,}")

print("\nPatients:", labeled_df["patient_id"].nunique())
print("Visits  :", labeled_df["checkup_id"].nunique())

# ------------------------------------------------------------
# SPLIT SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("PATIENT-LEVEL SPLIT")
print("=" * 100)

split_summary = (
    labeled_df.groupby("split")
    .agg(
        patients=("patient_id", "nunique"),
        visits=("checkup_id", "nunique"),
    )
    .reindex(["train", "validation", "test"])
)

display(split_summary)

# ------------------------------------------------------------
# LABEL SUMMARY
# ------------------------------------------------------------

label_columns = [
    "label_gingivitis",
    "label_class_ii_malocclusion",
    "label_dental_crowding",
    "label_tooth_structure_loss",
    "label_dental_caries",
    "label_convex_profile",
    "label_mandibular_skeletal_asymmetry",
    "label_periodontitis",
    "label_class_iii_malocclusion",
    "label_pulpitis",
    "label_deep_overbite",
    "label_class_i_malocclusion",
    "label_tooth_loss",
]

print("\n" + "=" * 100)
print("13-CLASS LABEL DISTRIBUTION")
print("=" * 100)

label_summary = pd.DataFrame({
    "label": label_columns,
    "positive_visits": [
        int(labeled_df[col].sum())
        for col in label_columns
    ],
    "positive_patients": [
        int(
            labeled_df.loc[
                labeled_df[col] == 1,
                "patient_id"
            ].nunique()
        )
        for col in label_columns
    ],
})

label_summary["visit_prevalence_%"] = (
    label_summary["positive_visits"]
    / len(labeled_df)
    * 100
)

display(
    label_summary.style.format({
        "positive_visits": "{:,}",
        "positive_patients": "{:,}",
        "visit_prevalence_%": "{:.2f}%"
    })
)

# ------------------------------------------------------------
# FINAL DATASET SAMPLE
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("FINAL DATASET — KEY COLUMNS")
print("=" * 100)

preview_columns = [
    "id",
    "checkup_id",
    "patient_id",
    "checkup_time",
    "photographs",
    "radiographs",
    "anomalies_en",
    "reconstructed_label_count",
    "reconstructed_labels",
    "has_reconstructed_label",
    "split",
]

display(
    labeled_df[preview_columns].head(10)
)

# ------------------------------------------------------------
# LABELLED / UNLABELLED VISITS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("LABEL COVERAGE")
print("=" * 100)

coverage = pd.DataFrame({
    "category": [
        "Visits with >=1 reconstructed label",
        "Visits without reconstructed label",
    ],
    "visits": [
        int((labeled_df["has_reconstructed_label"] == 1).sum()),
        int((labeled_df["has_reconstructed_label"] == 0).sum()),
    ],
})

coverage["percentage_%"] = (
    coverage["visits"]
    / len(labeled_df)
    * 100
)

display(
    coverage.style.format({
        "visits": "{:,}",
        "percentage_%": "{:.2f}%"
    })
)

print("\nDONE")

Labeled dataset : /home/ubuntu/Projects/thesis-code/results/labeled_patient_level_dataset/labeled_dataset.csv
Exists          : True

FINAL LABELED PATIENT-LEVEL DATASET
Rows    : 8,775
Columns : 51

Patients: 4800
Visits  : 8775

PATIENT-LEVEL SPLIT


,patients,visits
split,,
train,3360,6129
validation,720,1330
test,720,1316



13-CLASS LABEL DISTRIBUTION


,label,positive_visits,positive_patients,visit_prevalence_%
0,label_gingivitis,"1,987","1,691",22.64%
1,label_class_ii_malocclusion,"1,825","1,498",20.80%
2,label_dental_crowding,"1,148","1,066",13.08%
3,label_tooth_structure_loss,"1,042",831,11.87%
4,label_dental_caries,955,810,10.88%
5,label_convex_profile,746,739,8.50%
6,label_mandibular_skeletal_asymmetry,524,524,5.97%
7,label_periodontitis,669,553,7.62%
8,label_class_iii_malocclusion,462,389,5.26%
9,label_pulpitis,443,364,5.05%



FINAL DATASET — KEY COLUMNS


,id,checkup_id,patient_id,checkup_time,photographs,radiographs,anomalies_en,reconstructed_label_count,reconstructed_labels,has_reconstructed_label,split
0,1,0001-001,1,22/08/2022 13:00,0001-001-01.jpg,0001-001-01.jpg,"Post And Core Crown,Dental Fluorosis,Tooth Str...",3,Tooth Structure Loss|Dental Caries|Pulpitis,1,train
1,2,0001-002,1,30/09/2022 17:00,0001-002-01.jpg,NaN,NaN,0,NaN,0,train
2,3,0001-003,1,12/06/2023 10:00,"0001-003-01.jpg,0001-003-02.jpg,0001-003-03.jp...",0001-003-01.jpg,"Dental Fluorosis,Tooth Wear,Enamel Hypoplasia,...",1,Dental Caries,1,train
3,4,0001-004,1,20/06/2023 10:00,"0001-004-01.jpg,0001-004-02.jpg",NaN,"Tooth Wear,Enamel Hypoplasia,Dental Caries",1,Dental Caries,1,train
4,5,0001-005,1,25/06/2023 10:00,"0001-005-01.jpg,0001-005-02.jpg,0001-005-03.jp...",0001-005-01.jpg,Gingivitis,1,Gingivitis,1,train
5,6,0002-001,2,02/01/2024 14:00,0002-001-01.jpg,0002-001-01.jpg,"Periapical Periodontitis,Cracked Tooth",1,Periodontitis,1,test
6,7,0003-001,3,26/05/2020 17:30,"0003-001-01.jpg,0003-001-02.jpg,0003-001-03.jp...",NaN,NaN,0,NaN,0,test
7,8,0003-002,3,01/12/2023 16:30,"0003-002-01.jpg,0003-002-02.jpg",NaN,Tooth Structure Loss,1,Tooth Structure Loss,1,test
8,9,0004-001,4,19/01/2024 10:30,NaN,NaN,Dental Caries,1,Dental Caries,1,train
9,10,0004-002,4,26/01/2024 14:00,NaN,NaN,Tooth Structure Loss,1,Tooth Structure Loss,1,train



LABEL COVERAGE


,category,visits,percentage_%
0,Visits with >=1 reconstructed label,"7,256",82.69%
1,Visits without reconstructed label,"1,519",17.31%



DONE


# 05 — Final Labeled Patient-Level Dataset & Benchmark Readiness

## 1. Objective

The objective of this milestone was to construct a final, reproducible, patient-level dataset suitable for downstream classification experiments.

The process combined:

* the original COde dataset,
* patient-level split assignment,
* label reconstruction from `anomalies_en`,
* selection of sufficiently frequent labels,
* and attachment of reconstructed labels to the authoritative patient-level split.

The resulting artifact is intended to serve as the fixed dataset for the subsequent **Baseline Classification** milestone.

---

## 2. Final Dataset

The final dataset is:

```text
results/labeled_patient_level_dataset/labeled_dataset.csv
```

Dataset characteristics:

| Property                           |  Value |
| ---------------------------------- | -----: |
| Total visits                       |  8,775 |
| Unique patients                    |  4,800 |
| Total columns                      |     51 |
| Selected labels                    |     13 |
| Visits with ≥1 reconstructed label |  7,256 |
| Visits without reconstructed label |  1,519 |
| Label coverage                     | 82.69% |

The final dataset retains the original clinical, textual, photographic, and radiographic information together with the reconstructed binary label columns and patient-level split assignment.

---

## 3. Patient-Level Split

The patient-level split previously constructed in:

```text
results/patient_level_split/patient_split.csv
```

is the authoritative split for all downstream experiments.

The split contains:

| Split      |  Patients |    Visits |
| ---------- | --------: | --------: |
| Train      |     3,360 |     6,129 |
| Validation |       720 |     1,330 |
| Test       |       720 |     1,316 |
| **Total**  | **4,800** | **8,775** |

No new random split is introduced during this milestone.

All visits belonging to the same patient remain within the same split.

---

## 4. Leakage Validation

Patient-level separation and cross-split leakage were previously validated.

The final split satisfies:

```text
Patient overlap:      0
Visit overlap:        0
Photograph overlap:   0
Radiograph overlap:   0
Assignment validation: PASS
Overall status:        SAFE
```

Therefore, the final labeled dataset preserves the required patient-level isolation between training, validation, and test sets.

The existing leakage-audit artifacts remain available under:

```text
results/patient_level_split_leakage/
```

---

## 5. Label Reconstruction

The original dataset did not provide the required benchmark labels in a directly usable binary-label format.

Labels were therefore reconstructed from:

```text
anomalies_en
```

An atomic label inventory was first performed to determine the frequency and patient coverage of available labels.

Based on frequency and suitability for classification, the 13 most suitable labels were selected.

The reconstruction pipeline produces:

* binary label columns,
* the number of reconstructed labels per visit,
* the reconstructed label names,
* and an indicator showing whether a visit contains at least one selected label.

The reconstruction process is implemented in:

```text
src/data/label_reconstruction.py
```

and the resulting labeled patient-level dataset is generated by:

```text
src/data/labeled_patient_level_dataset.py
```

---

## 6. Selected 13 Labels

The final benchmark contains the following labels:

1. Gingivitis
2. Class II Malocclusion
3. Dental Crowding
4. Tooth Structure Loss
5. Dental Caries
6. Convex Profile
7. Mandibular Skeletal Asymmetry
8. Periodontitis
9. Class III Malocclusion
10. Pulpitis
11. Deep Overbite
12. Class I Malocclusion
13. Tooth Loss

These labels were retained because they provide substantially more samples than the lower-frequency atomic labels and therefore offer a more practical basis for baseline classification experiments.

---

## 7. Final Label Distribution

The resulting positive-visit distribution is:

| Rank | Label                         | Positive Visits | Positive Patients | Visit Prevalence |
| ---: | ----------------------------- | --------------: | ----------------: | ---------------: |
|    1 | Gingivitis                    |           1,987 |             1,691 |           22.64% |
|    2 | Class II Malocclusion         |           1,825 |             1,498 |           20.80% |
|    3 | Dental Crowding               |           1,148 |             1,066 |           13.08% |
|    4 | Tooth Structure Loss          |           1,042 |               831 |           11.87% |
|    5 | Dental Caries                 |             955 |               810 |           10.88% |
|    6 | Convex Profile                |             746 |               739 |            8.50% |
|    7 | Periodontitis                 |             669 |               553 |            7.62% |
|    8 | Mandibular Skeletal Asymmetry |             524 |               524 |            5.97% |
|    9 | Class III Malocclusion        |             462 |               389 |            5.26% |
|   10 | Pulpitis                      |             443 |               364 |            5.05% |
|   11 | Deep Overbite                 |             421 |               412 |            4.80% |
|   12 | Class I Malocclusion          |             359 |               332 |            4.09% |
|   13 | Tooth Loss                    |             309 |               268 |            3.52% |

The labels have different prevalence levels, including relatively rare classes such as Tooth Loss and Class I Malocclusion.

This class imbalance will therefore be considered during baseline evaluation and metric selection.

---

## 8. Multi-Label Structure

The reconstructed task is inherently **multi-label** rather than single-label multiclass classification.

A single visit may contain multiple selected labels.

For example, a visit may simultaneously contain:

```text
Tooth Structure Loss
Dental Caries
Pulpitis
```

Therefore, each of the 13 labels is represented as an independent binary target:

```text
label_gingivitis
label_class_ii_malocclusion
label_dental_crowding
...
label_tooth_loss
```

The downstream baseline will consequently evaluate the 13 classification targets independently while preserving the underlying multi-label structure.

---

## 9. Dataset Coverage

The final reconstruction provides:

```text
Visits with ≥1 selected label:     7,256
Visits without selected label:     1,519

Coverage:                           82.69%
```

The remaining visits are retained in the final dataset because they are part of the original patient-level benchmark population.

Whether unlabeled visits are used for a particular baseline experiment will be determined by the task definition of that experiment rather than by removing them from the canonical dataset.

---

## 10. Benchmark Readiness

The final dataset is considered ready for downstream baseline classification because:

* patient identities are isolated across splits;
* the authoritative patient-level split is fixed;
* cross-split leakage has been validated;
* the target labels have been reconstructed;
* 13 sufficiently frequent labels have been selected;
* the multi-label nature of the task has been explicitly preserved;
* label prevalence has been quantified;
* the final dataset is reproducibly generated by project scripts.

The canonical benchmark artifact is:

```text
results/labeled_patient_level_dataset/labeled_dataset.csv
```

Supporting artifacts are:

```text
results/labeled_patient_level_dataset/dataset_summary.json
results/labeled_patient_level_dataset/label_split_distribution.csv
```

---

## 11. Reproducibility

The final labeled patient-level dataset can be regenerated using:

```bash
python src/data/labeled_patient_level_dataset.py --force
```

The generation pipeline uses the previously established patient-level split and does not create a new random split.

The resulting artifact should therefore be treated as the fixed input dataset for subsequent baseline experiments.

---

## 12. Final Decision

This milestone is considered **COMPLETE**.

The dataset preparation pipeline has progressed from the original COde dataset to a validated, patient-level, labeled benchmark dataset:

```text
COde Raw Dataset
        ↓
Dataset / Modality Audit
        ↓
Patient Trace & Duplication Audit
        ↓
Patient-Level Split
        ↓
Split Leakage Audit
        ↓
Label Reconstruction
        ↓
13-Class Label Selection
        ↓
Patient-Level Label Attachment
        ↓
Final Labeled Patient-Level Dataset
        ↓
Baseline Classification
```

### Final status

**Dataset preparation: COMPLETE ✅**

**Patient-level split: FIXED ✅**

**Leakage validation: SAFE ✅**

**Label reconstruction: COMPLETE ✅**

**Benchmark dataset: READY ✅**

**Next milestone: Baseline Classification**
